<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/ml/notebooks/c5_l6.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C5-L6 · Stacking con OOF
Bases diversas + meta entrenado con predicciones honestas (cada tramo predicho a ciegas): si el stack no supera a la mejor base en OOS, manda lo simple.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/ml/data/c5_l6.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c5_l6.csv'), Path('data/c5_l6.csv'), Path('c5_l6.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
df['ret'] = df['close'].pct_change()
df['rango'] = (df['high']-df['low'])/df['close']
df['vol_z'] = (df['volumen']-df['volumen'].rolling(10).mean())/df['volumen'].rolling(10).std()
for k in (1, 2, 3, 5):
    df[f'lag_{k}'] = df['ret'].shift(k)
df['mom5'] = df['close']/df['close'].shift(5) - 1
data = df.dropna().reset_index(drop=True)
data['y'] = (data['ret'].shift(-1) > 0).astype(int)
data = data.iloc[:-1].reset_index(drop=True)
feat = ['rango','vol_z','lag_1','lag_2','lag_3','lag_5','mom5']
print('filas:', len(data), 'features:', feat)
assert data[feat].isna().sum().sum() == 0 and len(data) > 40

In [ ]:
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import KFold
X = data[feat].values; y = data['y'].values
split = int(len(data)*0.7)
X_tr, y_tr = X[:split], y[:split]
X_te, y_te = X[split:], y[split:]
kf = KFold(n_splits=5, shuffle=False)  # sin shuffle: orden temporal sagrado
oof = np.zeros((len(X_tr), 2))
bases = [RidgeClassifier(), HistGradientBoostingClassifier(max_iter=100)]
for tr, va in kf.split(X_tr):
    for j, b in enumerate(bases):
        b.fit(X_tr[tr], y_tr[tr])
        oof[va, j] = b.predict(X_tr[va])
meta = LogisticRegression().fit(oof, y_tr)
for b in bases: b.fit(X_tr, y_tr)
p1 = bases[0].predict(X_te); p2 = bases[1].predict(X_te)
p_meta = meta.predict(np.c_[p1, p2])
acc = lambda t, p: float((t == p).mean())
a1, a2, am = acc(y_te, p1), acc(y_te, p2), acc(y_te, p_meta)
print(f'Ridge={a1:.3f}  GBM={a2:.3f}  STACK={am:.3f}')

In [ ]:
m = len(y_te)//2
s1 = float((y_te[:m] == p_meta[:m]).mean()); s2 = float((y_te[m:] == p_meta[m:]).mean())
print(f'stack mitad1={s1:.3f} mitad2={s2:.3f} | mejor base={max(a1, a2):.3f}')

In [ ]:
assert oof.shape == (len(X_tr), 2)
assert set(np.unique(oof)).issubset({0.0, 1.0})
assert all(np.isfinite([a1, a2, am])) and 0.3 <= am <= 0.8
assert len(p_meta) == len(y_te)
print(f'OK L6: stacking OOF verificado, STACK={am:.3f} vs mejor base={max(a1, a2):.3f}')